# Building a Spam Filter with Naive Bayes

In this project, we're building a **spam filter** for SMS message using **multinomial Naive Bayes**.

To classify messages as spam or non-spam:

- algorithm will learns how humans classify messages.
  
- Uses that human knowledge to estimate probabilities for new messages — probabilities for spam and non-spam.
  
- Classifies a new message based on these probability values — if the probability for spam is greater, then it classifies the message as spam. Otherwise, it classifies it as non-spam (if the two probability values are equal, then we may need a human to classify the message).

we'll use the multinomial Naive Bayes algorithm along with a dataset of `5,572` SMS messages that are already classified by humans.

The dataset was put together by Tiago A. Almeida and José María Gómez Hidalgo, and it can be downloaded from the [The UCI Machine Learning Repository](https://dq-content.s3.amazonaws.com/433/SMSSpamCollection). The data collection process is described in more details on [this page](http://www.dt.fee.unicamp.br/~tiago/smsspamcollection/#composition), where you can also find some of the authors' papers.

**Note that due to the nature of spam messages, the dataset contains content that may be offensive to some users.**

## Exploring the Dataset

In [1]:
# import necessary module
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the Dataset
spam_dataset = pd.read_csv('../../dataset/SMSSpamCollection', sep = '\t', header = None, names=['Label', 'SMS'])                                                                             

/Users/bikashkumarsah/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Finding the number of rows and column in the dataset
spam_dataset.shape

(5572, 2)

In [3]:
# Find what percentage of message is spam and what percentage is ham
spam_dataset['Label'].value_counts(normalize = True) * 100

Label
ham     86.593683
spam    13.406317
Name: proportion, dtype: float64

We find that `86.6%` of message are ham(i.e non-spam) and the rest `13.4%` are spam message in the dataset.

## Training and Test Set

To test the spam filter, we're first going to split our dataset into two categories:

- A **training set**, which we'll use to "train" the computer how to classify messages.
- A **test set**, which we'll use to test how good the spam filter is with classifying new messages.


We're going to keep `80%` of our dataset for training, and `20%` for testing (we want to train the algorithm on as much data as possible, but we also want to have enough test data). The dataset has `5,572` messages, which means that:

- The training set will have `4,458` messages (about 80% of the dataset).
- The test set will have `1,114` messages (about 20% of the dataset).

our goal is to create a spam filter that classifies new messages with an accuracy greater than `80%` — so we expect that more than `80%` of the new messages will be classified correctly as spam or ham (non-spam).


In [4]:
# randomizing the entire dataset by using DataFrame.sample() method.
randomized_data = spam_dataset.sample(frac = 1, random_state = 1)

# Split the randomized dataset into a training and a test set.
training_set_index = round(len(randomized_data) * 0.8)
training_set = randomized_data[:training_set_index].reset_index(drop = True)
test_set = randomized_data[training_set_index:].reset_index(drop = True)

In [5]:
# percentage of spam and ham in both the training and test set
print(training_set['Label'].value_counts(normalize = True) * 100)
print(test_set['Label'].value_counts(normalize = True) * 100)

Label
ham     86.54105
spam    13.45895
Name: proportion, dtype: float64
Label
ham     86.804309
spam    13.195691
Name: proportion, dtype: float64


The percentage of spam and ham in both training and test set are similar which confirm a proper split of dataset.

## Data cleaning

To calculate all these probabilities, we'll first need to perform a bit of data cleaning to bring the data in a format that will allow us to extract easily all the information we need.

Essentially, we want to bring data to this format:

![img](https://dq-content.s3.amazonaws.com/433/cpgp_dataset_3.png)

Let's begin the data cleaning process by removing the punctuation and bringing all the words to lower case.


In [6]:
# Remove all the punctuation from the SMS columns.
# Transform every letter in every word to lower case.
training_set['SMS'] = training_set['SMS'].str.replace('\W', ' ', regex = True).str.lower()
test_set['SMS'] = test_set['SMS'].str.replace('\W', ' ', regex = True).str.lower()

# Check the first 5 value 
training_set['SMS'].head()

0                         yep  by the pretty sculpture
1        yes  princess  are you going to make me moan 
2                           welp apparently he retired
3                                              havent 
4    i forgot 2 ask ü all smth   there s a card on ...
Name: SMS, dtype: object

## Creating the Vocabulary

Let's create a list with all of the unique words that occur in the messages of our training set.

In [7]:
# Transform SMS column into a list by splitting the string at the space character.

training_set['SMS'] = training_set['SMS'].str.split(' ')

# Initialize an empty list.
vocabulary = list()

for row in training_set['SMS']:
    for word in row:
        vocabulary.append(word)

# Remove the duplicate from the vocabulary
vocabulary = set(vocabulary)
vocabulary = list(vocabulary)

In [8]:
print(vocabulary[:5])

['', 'roger', '09064012160', 'sickness', 'wicklow']


## The Final Training Set

In [9]:
# create the dictionary we need for our training set

word_counts_per_sms = {unique_word: [0] * len(training_set['SMS']) for unique_word in vocabulary}

for index, sms in enumerate(training_set['SMS']):
    for word in sms:
        word_counts_per_sms[word][index] += 1

# Transform word_counts_per_sms into a DataFrame using pd.DataFrame().
word_counts = pd.DataFrame(word_counts_per_sms)


In [10]:

# Concatenate the DataFrame we just built above with the DataFrame containing the training set (this way, we'll also have the Label and the SMS columns)
final_training_set = pd.concat([training_set, word_counts], axis = 1)
final_training_set.head()

,Label,SMS,,roger,09064012160,sickness,wicklow,145,yar,spatula,...,hype,16,stores,mi,8th,notebook,3optical,nag,canary,fast
0,ham,"[yep, , by, the, pretty, sculpture]",1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ham,"[yes, , princess, , are, you, going, to, make,...",3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ham,"[welp, apparently, he, retired]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ham,"[havent, ]",1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ham,"[i, forgot, 2, ask, ü, all, smth, , , there, s...",7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Calculating Constants First

We're now done with cleaning the training set, and we can begin creating the spam filter. The Naive Bayes algorithm will need to answer these two probability questions to be able to classify new messages:

\begin{equation}
P(Spam | w_1,w_2, ..., w_n) \propto P(Spam) \cdot \prod_{i=1}^{n}P(w_i|Spam)
\end{equation}

\begin{equation}
P(Ham | w_1,w_2, ..., w_n) \propto P(Ham) \cdot \prod_{i=1}^{n}P(w_i|Ham)
\end{equation}


Also, to calculate P(w<sub>i</sub>|Spam) and P(w<sub>i</sub>|Ham) inside the formulas above, we'll need to use these equations:

\begin{equation}
P(w_i|Spam) = \frac{N_{w_i|Spam} + \alpha}{N_{Spam} + \alpha \cdot N_{Vocabulary}}
\end{equation}

\begin{equation}
P(w_i|Ham) = \frac{N_{w_i|Ham} + \alpha}{N_{Ham} + \alpha \cdot N_{Vocabulary}}
\end{equation}


Some of the terms in the four equations above will have the same value for every new message. We can calculate the value of these terms once and avoid doing the computations again when a new messages comes in. Below, we'll use our training set to calculate:

- P(Spam) and P(Ham)
- N<sub>Spam</sub>, N<sub>Ham</sub>, N<sub>Vocabulary</sub>

We'll also use Laplace smoothing and set $\alpha = 1$.


In [21]:
# Calculate P(Spam) and P(Ham)

spam_message = final_training_set[final_training_set['Label'] == 'spam']
non_spam_message = final_training_set[final_training_set['Label'] == 'ham']

p_spam = len(spam_message) / len(final_training_set)
p_ham = len(non_spam_message) / len(final_training_set)

print(p_spam, p_ham)

0.13458950201884254 0.8654104979811574


In [19]:
# calculate N(spam), N(Ham), N(vocabulary)
n_spam = 0
n_ham = 0


spam_messages = final_training_set['SMS'][final_training_set['Label'] == 'spam']
for row in spam_messages:
    n_spam += len(row)

ham_messages = final_training_set['SMS'][final_training_set['Label'] == 'ham']
for row in ham_messages:
    n_ham += len(row)
# N_Vocabulary
n_vocabulary = len(vocabulary)

# Laplace smoothing
alpha = 1
print(n_spam, n_ham, n_vocabulary)

17956 71219 7784


## Calculating Parameters

Now that we have the constant terms calculated above, we can move on with calculating the parameters $P(w_i|Spam)$ and $P(w_i|Ham)$. Each parameter will thus be a conditional probability value associated with each word in the vocabulary.

The parameters are calculated using the formulas:

\begin{equation}
P(w_i|Spam) = \frac{N_{w_i|Spam} + \alpha}{N_{Spam} + \alpha \cdot N_{Vocabulary}}
\end{equation}

\begin{equation}
P(w_i|Ham) = \frac{N_{w_i|Ham} + \alpha}{N_{Ham} + \alpha \cdot N_{Vocabulary}}
\end{equation}

In [ ]:
# Initiate parameters
parameters_spam = {unique_word : 0 for unique_word in vocabulary}
parameters_ham = {unique_word : 0 for unique_word in vocabulary}

# Calculate parameters
for word in vocabulary:
    n_word_given_spam = spam_message[word].sum()   # spam_messages already defined in a cell above
    p_word_given_spam = (n_word_given_spam + alpha) / (n_spam + alpha*n_vocabulary)
    parameters_spam[word] = p_word_given_spam
    
    n_word_given_ham = non_spam_message[word].sum()   # non_spam_messages already defined in a cell above
    p_word_given_ham = (n_word_given_ham + alpha) / (n_ham + alpha*n_vocabulary)
    parameters_ham[word] = p_word_given_ham

## Classifying A New Message

Now that we have all our parameters calculated, we can start creating the spam filter. The spam filter can be understood as a function that:

- Takes in as input a new message (w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>).
- Calculates P(Spam|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>) and P(Ham|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>).
- Compares the values of P(Spam|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>) and P(Ham|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>), and:
    - If P(Ham|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>) > P(Spam|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>), then the message is classified as ham.
    - If P(Ham|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>) < P(Spam|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>), then the message is classified as spam.
    -  If P(Ham|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>) = P(Spam|w<sub>1</sub>, w<sub>2</sub>, ..., w<sub>n</sub>), then the algorithm may request human help.

In [30]:
import re

def classify(message):

    message = re.sub('\W', ' ', message)
    message = message.lower()
    message = message.split()

    '''    
    This is where we calculate:

    p_spam_given_message = ?
    p_ham_given_message = ?
    '''    
    p_spam_given_message = p_spam
    p_ham_given_message = p_ham

    for word in message:
        if word in parameters_spam:
            p_spam_given_message *= parameters_spam[word]
        if word in parameters_ham:
            p_ham_given_message *= parameters_ham[word]

            
    print('P(Spam|message):', p_spam_given_message)
    print('P(Ham|message):', p_ham_given_message)

    if p_ham_given_message > p_spam_given_message:
        print('Label: Ham')
    elif p_ham_given_message < p_spam_given_message:
        print('Label: Spam')
    else:
        print('Equal proabilities, have a human classify this!')        

In [31]:
message_1 = 'WINNER!! This is the secret code to unlock the money: C3421.'
message_2 = "Sounds good, Tom, then see u there"

classify(message_1)


P(Spam|message): 4.8441099043219405e-26
P(Ham|message): 3.3551837806850006e-28
Label: Spam


In [32]:
classify(message_2)

P(Spam|message): 1.099416001503406e-25
P(Ham|message): 9.431033496608998e-22
Label: Ham


## Measuring the Spam Filter's Accuracy

We'll now try to determine how well the spam filter does on our test set of 1,114 messages.

To make the measurement, we'll use **accuracy** as a metric:

Accuracy = number of correctly classified messages / total number of classified messages


In [33]:
def classify_test_set(message):

    message = re.sub('\W', ' ', message)
    message = message.lower()
    message = message.split()

    p_spam_given_message = p_spam
    p_ham_given_message = p_ham

    for word in message:
        if word in parameters_spam:
            p_spam_given_message *= parameters_spam[word]

        if word in parameters_ham:
            p_ham_given_message *= parameters_ham[word]

    if p_ham_given_message > p_spam_given_message:
        return 'ham'
    elif p_spam_given_message > p_ham_given_message:
        return 'spam'
    else:
        return 'needs human classification'

In [34]:
test_set['predicted'] = test_set['SMS'].apply(classify_test_set)
test_set.head()

,Label,SMS,predicted
0,ham,later i guess i needa do mcat study too,ham
1,ham,but i haf enuff space got like 4 mb,ham
2,spam,had your mobile 10 mths update to latest oran...,spam
3,ham,all sounds good fingers makes it difficult ...,ham
4,ham,all done all handed in don t know if mega sh...,ham


In [37]:
# Measure the accuracy of the spam filter.

correct =(test_set['Label'] == test_set['predicted']).sum()
total = len(test_set)
        
print('Correct:', correct)
print('Incorrect:', total - correct)
print('Accuracy:', correct/total)

Correct: 1099
Incorrect: 15
Accuracy: 0.9865350089766607


The accuracy is close to 98.65%, which is really good. Our spam filter looked at 1,114 messages that it hasn't seen in training, and classified 1,100 correctly.

## Conclusion

In this project, we managed to build a spam filter for SMS messages using the multinomial Naive Bayes algorithm. The filter had an accuracy of 98.65% on the test set, which is an excellent result. We initially aimed for an accuracy of over 80%, but we managed to do way better than that.